## 🔄 Recuperar el modelo

Google Colab borra toda su memoria RAM y sus archivos temporales cada vez que cerramos la pestaña. Si queremos usar nuestra IA, no tenemos que volver a entrenarla (lo cual tardaría horas), solo tenemos que **recuperarla**.

Para entender el código de abajo, imagina que es un nuevo día de clases y el pupitre está vacío. Tenemos que hacer 3 cosas:

1. **Abrir la mochila:** Conectamos nuestro Google Drive para acceder a los archivos guardados.
2. **Llamar al estudiante base:** Volvemos a descargar de internet el "cerebro" original (ej. Qwen o TinyLlama). Este cerebro sabe hablar, pero no sabe nada de nuestro SQL porque le borraron la memoria.
3. **Darle SUS apuntes (La Magia):** Cogemos nuestro archivo LoRA (el "cuaderno de notas" que guardamos ayer en Drive) y se lo pegamos al modelo base usando la función `PeftModel`.



> **💡 Nota de Data Engineer:** Fíjate que en este paso ya **NO** instalamos librerías como `trl` ni `datasets`, porque ya no estamos enseñando (Entrenamiento), solo estamos preguntando (Inferencia). Es un proceso mucho más ligero.
**Recuerda que debes utilizar la tarjeta grafica**

In [1]:
# 1. Instalamos las librerías básicas (esto siempre hay que hacerlo en un Colab nuevo)
!pip install -q transformers accelerate peft

from google.colab import drive
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 2. Abrimos la mochila (Conectamos Drive)
print("1. Conectando a Google Drive...")
drive.mount('/content/drive')

# 3. Llamamos al estudiante base (El cerebro original)
print("2. Descargando el cerebro base (Qwen)...")
modelo_base_id = "Qwen/Qwen2.5-Coder-1.5B"
tokenizer = AutoTokenizer.from_pretrained(modelo_base_id)
tokenizer.pad_token = tokenizer.eos_token

modelo_base = AutoModelForCausalLM.from_pretrained(
    modelo_base_id,
    device_map="auto",
    torch_dtype=torch.float16
)

# 4. Le entregamos SUS APUNTES desde tu Drive
print("3. Cargando TUS conocimientos desde Drive...")
ruta_mi_modelo = "/content/drive/MyDrive/modelo_qwen_sql_final"

# ¡ESTA ES LA LÍNEA MÁGICA! Fusiona el cerebro con tus apuntes
mi_ia_experta = PeftModel.from_pretrained(modelo_base, ruta_mi_modelo)

print("✅ ¡Modelo recuperado con éxito! Listo para trabajar.")

# --- 5. LA PRUEBA ---
print("\n--- Haciendo una consulta SQL ---")
pregunta = "Muestra los nombres de los clientes que han comprado más de 5 veces"
contexto = "CREATE TABLE clientes (nombre VARCHAR, compras INT)"
prompt = f"Pregunta: {pregunta}\nContexto: {contexto}\nSQL: "

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = mi_ia_experta.generate(
        **inputs, max_new_tokens=50, temperature=0.01, do_sample=False, eos_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

1. Conectando a Google Drive...
Mounted at /content/drive
2. Descargando el cerebro base (Qwen)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

3. Cargando TUS conocimientos desde Drive...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


✅ ¡Modelo recuperado con éxito! Listo para trabajar.

--- Haciendo una consulta SQL ---
Pregunta: Muestra los nombres de los clientes que han comprado más de 5 veces
Contexto: CREATE TABLE clientes (nombre VARCHAR, compras INT)
SQL:  SELECT nombre FROM clientes GROUP BY nombre HAVING COUNT(*) > 5
